# Provide Personalized HR Career Consultation for Job Seekers

As a product owner, I want to build and integrate a fine-tuned HR consultation model so that job seekers can receive personalized career advice instead of using a generic OpenAI model.

## Setup

### Install required libraries

In [ ]:
%%time
%pip install matplotlib==3.9.0 pandas==2.2.2 numpy==1.26.0 scikit-learn==1.5.1
%pip install torch --index-url https://download.pytorch.org/whl/cpu
%pip install transformers
   

### Import required libraries

The following imports the required libraries:

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import torch
from torch.nn import Module
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForCausalLM

# You can also use this section to suppress warnings generated by your code:
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')

### Defining helper functions

The following are some helper functions to help with plotting, saving, and loading files. 

In [ ]:
def plot(COST,ACC):
    fig, ax1 = plt.subplots()
    color = 'tab:red'
    ax1.plot(COST, color=color)
    ax1.set_xlabel('epoch', color=color)
    ax1.set_ylabel('total loss', color=color)
    ax1.tick_params(axis='y', color=color)

    ax2 = ax1.twinx()
    color = 'tab:blue'
    ax2.set_ylabel('accuracy', color=color)  # You already handled the x-label with ax1
    ax2.plot(ACC, color=color)
    ax2.tick_params(axis='y', color=color)
    fig.tight_layout()  # otherwise the right y-label is slightly clipped

    plt.show()


## Loading predefined model and tokenizer

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

# Dedicate device for inference
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)


## Preparation data

In [ ]:
DATA_PATH = "/home/vitaly/Documents/project/hire-tailor/train-models/dataset/hr-dataset-ext.json"

df = pd.read_json(DATA_PATH)
df = df['instructions']
df.head()


def format_prompt(dataset, eos_token):
    output_texts = []

    for i in range(len(dataset)):
        text = (
            f"### Instruction:\n{dataset.iloc[i]['instruction']}"
            f"\n\n### Response:\n{dataset.iloc[i]['output']}{eos_token}"
        )
        output_texts.append(text)
    return output_texts

def format_prompt_no_response(dataset, eos_token):
    output_texts = []
    for i in range(len(dataset)):
        text = (
            f"### Instruction:\n{dataset.iloc[i]['instruction']}"
            f"\n\n### Response:{eos_token}"
        )
        output_texts.append(text)
    return output_texts

# Split the dataset into training and testing sets
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)


### Implement training dataset and dataloader

In [ ]:
class HRDataset(Dataset):
    def __init__(self, instructions, tokenizer, max_length=512):
        self.full_prompt = format_prompt(instructions, tokenizer.eos_token)
        self.instruction_no_response = format_prompt_no_response(instructions, tokenizer.eos_token)
        self.tokenizer = tokenizer
        self.max_length = max_length
 
    def __len__(self):
        return len(self.full_prompt)

    def __getitem__(self, idx):
        instruction = self.full_prompt[idx]
        
        encoding = self.tokenizer(
            instruction,
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt'
        )
        input_ids = encoding['input_ids'].squeeze(0)
        attention_mask = encoding['attention_mask'].squeeze(0)

        instruction_no_response = self.instruction_no_response[idx]
        prompt_tokens = self.tokenizer(
            instruction_no_response,
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt'
        )["input_ids"].squeeze(0)

        labels = input_ids.clone()

        prompt_len = prompt_tokens.shape[0]
        labels[:prompt_len] = -100

        labels[attention_mask == 0] = -100

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels
        }

train_dataset = HRDataset(train_df[:5], tokenizer, max_length=96)
train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True)

print(f"Number of training samples: {len(train_dataset)}")

## LoRA (Low-Rank Adaptation)

In [ ]:
class LoRALayer(torch.nn.Module):
    def __init__(self, in_dim, out_dim, rank, alpha, dtype=torch.float16, dropout_prob=0.05):
        super().__init__()
        # Compute standart deviation for decreasing the scale of initial weights
        std_dev = 1 / torch.sqrt(torch.tensor(rank).float())
        # Initialize A and B matrices with appropriate scaling
        # A is initialized with random values scaled by std_dev, B is initialized with zeros
        self.A = torch.nn.Parameter(torch.randn(in_dim, rank, dtype=dtype) * std_dev).to(device)
        self.B = torch.nn.Parameter(torch.zeros(rank, out_dim, dtype=dtype)).to(device)
        # Scale the output of the LoRA layer by alpha divided by rank to maintain a consistent scale of updates
        self.scale = alpha / rank
        # Add dropout to the LoRA layer to prevent overfitting and improve generalization
        self.dropout = torch.nn.Dropout(dropout_prob)
    def forward(self, x):
        x = self.scale * (self.dropout(x) @ self.A @ self.B)
        return x


class LinearWithLoRA(torch.nn.Module):
    def __init__(self, linear, rank, alpha, dropout_prob=0.05):
        super().__init__()
        dtype = linear.weight.dtype
        self.linear = linear

        # Freeze original weights
        for param in self.linear.parameters():
            param.requires_grad = False

        self.lora = LoRALayer(
            linear.in_features, linear.out_features, rank, alpha, dtype, dropout_prob
        )

    def forward(self, x):
        return self.linear(x) + self.lora(x)

### Applying LoRA to model

In [ ]:
# Set the dropout probability for the LoRA layers
dropout_prob = 0.05
# Set the rank and alpha parameters for the LoRA layers
rank = 2
# Set the alpha parameter for the LoRA layers, which controls the scaling of the LoRA updates
alpha = 8

# Define the target modules for Lora application
# These are the names of the linear layers in the attention mechanism that we want to apply LoRA to.
target_modules = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj"
]

# Function to apply LoRA to the specified linear layers in the model
def apply_lora_to_model(model, target_modules, rank, alpha, dropout_prob):
    for name, module in model.named_modules():
        if any(target in name for target in target_modules) and isinstance(module, torch.nn.Linear):
            # Store the original linear layer
            base_linear = module
            # Initialize a new LinearWithLoRA layer using the original linear layer and the specified LoRA parameters
            lora_linear = LinearWithLoRA(base_linear, rank, alpha, dropout_prob)
            # Get the parent module of the original linear layer
            parent_module = dict(model.named_modules())[name.rsplit('.', 1)[0]]
            # Replace the original linear layer with the new LinearWithLoRA layer in the parent module
            setattr(parent_module, name.rsplit('.', 1)[-1], lora_linear)

# Apply LoRA to the model's linear layers specified in target_modules with the defined rank, alpha, and dropout probability
apply_lora_to_model(model, target_modules, rank=rank, alpha=alpha, dropout_prob=dropout_prob)

### Train model

In [ ]:
criterion = torch.nn.CrossEntropyLoss(ignore_index=-100)
optimizer = torch.optim.AdamW( [p for p in model.parameters() if p.requires_grad], lr=2e-4)
num_epochs = 10

def train(model, dataloader, criterion, optimizer, num_epochs, device):
    model.train()
    for epoch in range(num_epochs):
        total_loss = 0

        for step, batch in enumerate(dataloader):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
          
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)

            logits = outputs.logits

            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()

            loss = criterion(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1)
            )

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

            
        avg_loss = total_loss / len(dataloader)
        print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {avg_loss:.4f}")

train(model, train_loader, criterion, optimizer, num_epochs, device)

### Evaluation

In [ ]:

def evaluate(model, dataloader, device):
    model.eval()
    total_loss = 0
    total_correct = 0
    total_tokens = 0

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            logits = outputs.logits
            loss = outputs.loss

            total_loss += loss.item()


            shift_logits = logits[:, :-1, :]
            shift_labels = labels[:, 1:]

            predictions = shift_logits.argmax(dim=-1)

            mask = shift_labels != -100

            correct_predictions = (predictions == shift_labels) & mask

            total_correct += correct_predictions.sum().item()
            total_tokens += mask.sum().item()

    avg_loss = total_loss / len(dataloader)
    accuracy = total_correct / total_tokens if total_tokens > 0 else 0

    return avg_loss, accuracy

test_dataset = HRDataset(test_df, tokenizer, max_length=96)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False) 

avg_loss, accuracy = evaluate(model, test_loader, device)

print(f"Test Loss: {avg_loss:.4f}, Test Accuracy: {accuracy:.4f}")

